# Case Study: e-f Rabi on a shared drive port

A worked example between Lecture 1 and Lecture 2: an e-f Rabi experiment written in the plain DSL on a
real SHFQC description (compiled in emulation), where the g-e and the e-f drive share one physical output.

## Imports

In [1]:
from laboneq.simple import *
from laboneq.dsl.experiment.builtins import *
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

import numpy as np


## Generate setup

A real SHFQC, compiled in emulation mode - no hardware needed, but the real compiler is active

In [2]:
setup = DeviceSetup(uid="case_studies_setup")
setup.add_dataserver(host="localhost", port=8004)

setup.add_instruments(
    SHFQC(uid="device_shfqc", address="DEV12345", device_options="SHFQC/16W/LRT/PLUS/QC6CH/RTR", interface="1GBe")
)

setup.add_connections(
    "device_shfqc",
    create_connection(to_signal="q0/drive",    ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/drive_ef", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure",  ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire",  ports="QACHANNELS/0/INPUT"),
)

session = Session(device_setup=setup)
session.connect(do_emulation=True)


[2026.10.28 11:42:54.187] INFO    Logging initialized from [Default inline config in laboneq.laboneq_logging] logdir is /Users/emiliod/repositories/laboneq-lecture/laboneq_output/log
[2026.10.28 11:42:54.189] INFO    VERSION: laboneq 26.7.0
[2026.10.28 11:42:54.190] INFO    Connecting to data server at localhost:8004
[2026.10.28 11:42:54.192] INFO    Connected to Zurich Instruments LabOne Data Server version 26.07.2.5 at localhost:8004
[2026.10.28 11:42:54.194] INFO    Configuring the device setup
[2026.10.28 11:42:54.195] INFO    The device setup is configured


## Generate calibration for each signal lines

Structure is ready - fill in the frequencies together

In [3]:
calibration = Calibration()

drive_lo   = Oscillator(frequency=4e9)   # LO for the drive / drive_ef line
measure_lo = Oscillator(frequency=7e9)   # LO for the measure / acquire line

calibration["/logical_signal_groups/q0/drive"] = SignalCalibration(
    local_oscillator=drive_lo,
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.AUTO),   # g-e drive IF
    range=10,
)
calibration["/logical_signal_groups/q0/drive_ef"] = SignalCalibration(
    local_oscillator=Oscillator(frequency=4.2e9),
    oscillator=Oscillator(frequency=-100e6, modulation_type=ModulationType.AUTO),   # e-f drive IF
    range=10,
)
calibration["/logical_signal_groups/q0/measure"] = SignalCalibration(
    local_oscillator=measure_lo,
    oscillator=Oscillator(frequency=80e6, modulation_type=ModulationType.AUTO),
    range=-25,
)
calibration["/logical_signal_groups/q0/acquire"] = SignalCalibration(
    local_oscillator=measure_lo,
    oscillator=Oscillator(frequency=80e6, modulation_type=ModulationType.AUTO),
    range=10,
)

setup.set_calibration(calibration)


Before we sequence the two drive pulses on the same channel, a quick look at how LabOne Q schedules things: [Timing Rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html)

## e-f Rabi

In [ ]:
amplitudes = np.linspace(0.0, 1.0, 11)

ge_pulse = pulse_library.gaussian(uid="ge_pulse", length=23e-9, amplitude=0.7)
ef_pulse = pulse_library.gaussian(uid="ef_pulse", length=16e-9)
measure_pulse = pulse_library.const(uid="measure_pulse", length=2e-6, amplitude=0.5)
integration_kernel = pulse_library.const(uid="integration_kernel", length=2e-6)

@experiment(signals=[
    ExperimentSignal("drive",    map_to="/logical_signal_groups/q0/drive"),
    ExperimentSignal("drive_ef", map_to="/logical_signal_groups/q0/drive_ef"),
    ExperimentSignal("measure",  map_to="/logical_signal_groups/q0/measure"),
    ExperimentSignal("acquire",  map_to="/logical_signal_groups/q0/acquire"),
])
def ef_rabi(amplitudes):
    with acquire_loop_rt(uid="shots", count=10, averaging_mode=AveragingMode.CYCLIC, acquisition_type=AcquisitionType.INTEGRATION):
        with sweep(parameter=SweepParameter(uid="ef_amplitude", values=amplitudes)) as amplitude:

            # bring the qubit from |g> to |e>, then drive e-f with the swept amplitude
            # ...
            # measure the qubit
            with section(uid="qubit_measure", play_after="qubit_drive_ef"):
                play(signal="measure", pulse=measure_pulse)
                acquire(signal="acquire", handle="main", kernel=integration_kernel)

exp = ef_rabi(amplitudes)


In [5]:
compiled_experiment = session.compile(exp)


[2026.10.28 11:42:57.873] INFO    Starting LabOne Q Compiler run...
[2026.10.28 11:42:57.874] INFO    Resolved modulation type of oscillator on signal: 'acquire' to Software
[2026.10.28 11:42:57.874] INFO    Resolved modulation type of oscillator on signal: 'drive' to Hardware
[2026.10.28 11:42:57.875] INFO    Resolved modulation type of oscillator on signal: 'drive_ef' to Hardware
[2026.10.28 11:42:57.875] INFO    Resolved modulation type of oscillator on signal: 'measure' to Software
[2026.10.28 11:42:57.876] INFO    Schedule completed. [0.000 s]
[2026.10.28 11:42:57.897] INFO    Code generation completed. [0.021 s]
[2026.10.28 11:42:57.897] INFO    Completed compilation step 1 of 1. [0.022 s]
[2026.10.28 11:42:57.905] INFO     ───────────────────────────────────────────────────────────────────── 
[2026.10.28 11:42:57.905] INFO      Device            AWG   SeqC LOC   CT entries   Waveforms   Samples  
[2026.10.28 11:42:57.906] INFO     ────────────────────────────────────────────────

Plot the signals

Check the full experiment interactively!

In [6]:
show_pulse_sheet("ef_rabi", compiled_experiment)


[2026.10.28 11:42:59.729] INFO    Recompiling the experiment due to missing extra information in the compiled experiment. Compile with `OUTPUT_EXTRAS=True` and `MAX_EVENTS_TO_PUBLISH=1000` to bypass this step with a small impact on the compilation time.
[2026.10.28 11:42:59.730] INFO    Starting LabOne Q Compiler run...
[2026.10.28 11:42:59.730] INFO    Resolved modulation type of oscillator on signal: 'acquire' to Software
[2026.10.28 11:42:59.731] INFO    Resolved modulation type of oscillator on signal: 'drive' to Hardware
[2026.10.28 11:42:59.731] INFO    Resolved modulation type of oscillator on signal: 'drive_ef' to Hardware
[2026.10.28 11:42:59.732] INFO    Resolved modulation type of oscillator on signal: 'measure' to Software
[2026.10.28 11:42:59.733] INFO    Schedule completed. [0.000 s]
[2026.10.28 11:42:59.737] INFO    Code generation completed. [0.003 s]
[2026.10.28 11:42:59.739] INFO    Completed compilation step 1 of 1. [0.007 s]
[2026.10.28 11:42:59.742] INFO    Finishe

/Users/emiliod/repositories/laboneq-lecture/ef_rabi_2026-10-28-11-42-59.html